In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, subprocess, sys, time
import numpy as np
import pandas as pd

INPUT, WORK=Path('/kaggle/input'),Path('/kaggle/working')
if importlib.util.find_spec('numpyro') is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...",flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','numpyro==0.22.0'],check=True)
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC,NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
import matplotlib.pyplot as plt

print('JAX backend:',jax.default_backend(),'| devices:',jax.devices(),'| NumPyro:',numpyro.__version__)
if jax.default_backend()!='gpu':
    raise RuntimeError('GPU preflight failed: select GPU T4 x2, restart the session, and run again.')

def newest(pattern,label):
    paths=list(INPUT.rglob(pattern))
    if not paths: raise FileNotFoundError(f'Attach saved output containing {label}: {pattern}')
    return max(paths,key=lambda p:p.stat().st_mtime)

binary_sources=[]
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name!='binary': continue
    try: meta=json.loads(mp.read_text(encoding='utf-8'))
    except Exception: continue
    if meta.get('family')=='binary' and int(meta.get('parameter_draws_unique',0))>=100_000:
        binary_sources.append((meta.get('created_utc',''),mp.parent))
if not binary_sources: raise FileNotFoundError('Attach Day 7 Cosmic swan Bayesian binary simulations.')
_,BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
observations=np.load(BINARY_DIR/'observations.npy',mmap_mode='r')
parameters=pd.read_csv(BINARY_DIR/'parameters.csv')
time_days=np.asarray(np.load(BINARY_DIR/'time_days.npy'),dtype=float)
if observations.ndim!=2 or observations.shape!=(len(parameters),len(time_days)):
    raise ValueError('Day 7 observations, parameters, and time axis do not match.')

DIAG_PATH=newest('day9_12_conditional_mle_diagnostics.csv','Day 9.12 conditional MLE diagnostics')
PARAM_PATH=newest('day9_12_conditional_mle_parameters.csv','Day 9.12 conditional MLE parameter estimates')
case_table=pd.read_csv(DIAG_PATH)
parameter_table=pd.read_csv(PARAM_PATH)
needed_diag={'simulation_row','optimizer_success','period_fixed_days'}
if not needed_diag.issubset(case_table.columns):
    raise ValueError(f'Day 9.12 diagnostics missing columns: {sorted(needed_diag-set(case_table.columns))}')
CASE_ROWS=sorted(case_table.loc[case_table.optimizer_success.astype(bool),'simulation_row'].astype(int).tolist())
if len(CASE_ROWS)!=3: raise ValueError(f'Expected 3 successful Day 9.12 refinement cases; found {CASE_ROWS}')
PERIOD_FIXED=case_table.set_index('simulation_row').period_fixed_days.astype(float).to_dict()
SEEDS={}
for row in CASE_ROWS:
    sub=parameter_table[parameter_table.simulation_row.astype(int).eq(row)]
    if sub.parameter.duplicated().any(): raise ValueError(f'Duplicate Day 9.12 parameter values for row {row}.')
    vals=sub.set_index('parameter').estimate.astype(float).to_dict()
    required={'baseline_flux','phase','depth_primary','depth_ratio','duration_fraction',
              'ellipsoidal_amplitude','reflection_amplitude','noise_sigma'}
    missing=required-set(vals)
    if missing or any(not np.isfinite(vals[k]) for k in required if k in vals):
        raise ValueError(f'Invalid or missing MLE seeds for row {row}: {sorted(missing)}')
    if not np.isfinite(PERIOD_FIXED[row]) or not 0.1<=PERIOD_FIXED[row]<=13.0 or row>=len(parameters):
        raise ValueError(f'Invalid fixed period or simulation row: {row}')
    SEEDS[row]=vals

OUT=WORK/('day9-13-mle-seeded-nuts-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)
CHAINS,WARMUP,SAMPLES=2,400,500
TARGET_ACCEPT,MAX_TREE_DEPTH=0.90,10
CHAIN_METHOD='parallel' if len(jax.devices())>=2 else 'sequential'
print('[PASS] Day 7 simulations:',BINARY_DIR)
print('[PASS] Day 9.12 MLE fits:',DIAG_PATH.parent)
print('Rows:',CASE_ROWS)
print('Fixed periods:',{r:PERIOD_FIXED[r] for r in CASE_ROWS})
print(f'NUTS: {CHAINS} chains x {WARMUP} warmup + {SAMPLES} draws; dense mass; target_accept={TARGET_ACCEPT}; max_tree_depth={MAX_TREE_DEPTH}')
print('Output:',OUT)

In [ ]:
def smooth_clip_jax(x,sharpness=500.0):
    return (jax.nn.softplus(sharpness*x)-jax.nn.softplus(sharpness*(x-1.0)))/sharpness

def predict_jax(t,period,phase,depth,ratio,duration,ellip,reflection,baseline):
    orb=jnp.mod(t/period-phase+0.5,1.0)-0.5
    sec=jnp.mod(orb,1.0)-0.5
    half=jnp.maximum(duration/2.0,1e-6)
    ingress=jnp.maximum(half*0.18,1e-6)
    primary=smooth_clip_jax((half+ingress-jnp.abs(orb))/(2.0*ingress))
    secondary=smooth_clip_jax((half+ingress-jnp.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary+ellip*jnp.cos(4*jnp.pi*orb)+reflection*jnp.cos(2*jnp.pi*orb)

case_rows=[]
parameter_rows=[]
interval_rows=[]
plot_rows=[]
for row in CASE_ROWS:
    started=time.time()
    seeds=SEEDS[row]
    period=float(PERIOD_FIXED[row])
    t=np.asarray(time_days,dtype=float)
    y=np.asarray(observations[row],dtype=float)
    valid=np.isfinite(t)&np.isfinite(y)
    t,y=t[valid],y[valid]
    order=np.argsort(t)
    t,y=t[order],y[order]
    t=t-t.min()
    phase0=float(seeds['phase']%1.0)
    time_j,obs_j=jnp.asarray(t),jnp.asarray(y)

    def model():
        baseline=numpyro.sample('baseline_flux',dist.Normal(1.0,0.02))
        phase_offset=numpyro.sample('phase_offset',dist.Normal(0.0,0.025))
        depth=numpyro.sample('depth_primary',dist.Uniform(0.002,0.45))
        ratio=numpyro.sample('depth_ratio',dist.Uniform(0.0,1.0))
        duration=numpyro.sample('duration_fraction',dist.Uniform(0.005,0.12))
        ellip=numpyro.sample('ellipsoidal_amplitude',dist.Uniform(0.0,0.02))
        reflection=numpyro.sample('reflection_amplitude',dist.Uniform(-0.01,0.01))
        log_noise=numpyro.sample('log_noise',dist.Uniform(np.log(1e-4),np.log(0.03)))
        phase=jnp.mod(phase0+phase_offset,1.0)
        pred=predict_jax(time_j,period,phase,depth,ratio,duration,ellip,reflection,baseline)
        numpyro.sample('flux',dist.Normal(pred,jnp.exp(log_noise)),obs=obs_j)

    init={'baseline_flux':float(np.clip(seeds['baseline_flux'],0.9501,1.0499)),
        'phase_offset':0.0,
        'depth_primary':float(np.clip(seeds['depth_primary'],0.002001,0.449999)),
        'depth_ratio':float(np.clip(seeds['depth_ratio'],1e-5,0.99999)),
        'duration_fraction':float(np.clip(seeds['duration_fraction'],0.005001,0.119999)),
        'ellipsoidal_amplitude':float(np.clip(seeds['ellipsoidal_amplitude'],1e-5,0.01999)),
        'reflection_amplitude':float(np.clip(seeds['reflection_amplitude'],-0.00999,0.00999)),
        'log_noise':float(np.log(np.clip(seeds['noise_sigma'],1.0001e-4,0.02999)))}
    latent=('baseline_flux','phase_offset','depth_primary','depth_ratio','duration_fraction',
            'ellipsoidal_amplitude','reflection_amplitude','log_noise')
    kernel=NUTS(model,init_strategy=init_to_value(values=init),target_accept_prob=TARGET_ACCEPT,
                max_tree_depth=MAX_TREE_DEPTH,dense_mass=[latent])
    mcmc=MCMC(kernel,num_warmup=WARMUP,num_samples=SAMPLES,num_chains=CHAINS,
              chain_method=CHAIN_METHOD,progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(130000+row),extra_fields=('diverging','accept_prob','num_steps'))
        grouped=mcmc.get_samples(group_by_chain=True)
        flat=mcmc.get_samples(group_by_chain=False)
        diag=numpyro_summary(grouped,group_by_chain=True)
        per_param=[]
        for name,item in diag.items():
            rhat_arr=np.asarray(item.get('r_hat',np.nan),dtype=float)
            ess_arr=np.asarray(item.get('n_eff',np.nan),dtype=float)
            rhat=float(np.nanmax(rhat_arr)) if np.isfinite(rhat_arr).any() else float('nan')
            ess=float(np.nanmin(ess_arr)) if np.isfinite(ess_arr).any() else float('nan')
            per_param.append((name,rhat,ess))
        extras=mcmc.get_extra_fields(group_by_chain=True)
        divergences=int(np.asarray(extras['diverging']).sum())
        steps=np.asarray(extras['num_steps'],dtype=float)
        accept=np.asarray(extras['accept_prob'],dtype=float)
        max_tree_fraction=float(np.mean(steps >= (2**MAX_TREE_DEPTH-1)))
        rhats=[v[1] for v in per_param if np.isfinite(v[1])]
        esses=[v[2] for v in per_param if np.isfinite(v[2])]
        max_rhat=max(rhats) if rhats else float('nan')
        min_ess=min(esses) if esses else float('nan')
        status='screen_pass' if len(rhats)==len(per_param) and len(esses)==len(per_param) and max_rhat<=1.05 and min_ess>=100 and divergences==0 else 'needs_review'

        transformed={k:np.asarray(v) for k,v in grouped.items()}
        transformed['phase']=(phase0+transformed['phase_offset'])%1.0
        transformed['noise_sigma']=np.exp(transformed['log_noise'])
        np.savez_compressed(OUT/f'conditional_row_{row}_posterior_by_chain.npz',**transformed)
        truth=parameters.iloc[row]
        truth_values={name:float(truth[name]) for name in ['phase','depth_primary','depth_ratio','duration_fraction',
            'ellipsoidal_amplitude','reflection_amplitude','noise_sigma']}
        truth_values['baseline_flux']=1.0
        phase_truth=(truth_values['phase']-phase0+0.5)%1.0-0.5
        for name,rhat,ess in per_param:
            parameter_rows.append({'simulation_row':row,'parameter':name,'r_hat':rhat,'effective_sample_size':ess})
        for name in ['baseline_flux','phase_offset','depth_primary','depth_ratio','duration_fraction',
                     'ellipsoidal_amplitude','reflection_amplitude','noise_sigma']:
            vals=transformed[name] if name!='phase_offset' else transformed[name]
            q05,med,q95=np.quantile(vals,[0.05,0.5,0.95])
            truth_value=phase_truth if name=='phase_offset' else truth_values[name]
            interval_rows.append({'simulation_row':row,'parameter':name,'period_fixed_days':period,
                'q05':float(q05),'median':float(med),'q95':float(q95),
                'known_truth_evaluation_only':float(truth_value),
                'truth_inside_90pct_interval':bool(q05<=truth_value<=q95),
                'scope':'conditional_on_profile_period'})
        med={k:np.median(v,axis=(0,1)) for k,v in transformed.items()}
        phase_fit=float((phase0+float(med['phase_offset']))%1.0)
        pred=np.asarray(predict_jax(time_j,period,phase_fit,float(med['depth_primary']),float(med['depth_ratio']),
            float(med['duration_fraction']),float(med['ellipsoidal_amplitude']),float(med['reflection_amplitude']),
            float(med['baseline_flux'])),dtype=float)
        plot_rows.append((row,t,y,pred,period))
        case_rows.append({'simulation_row':row,'status':status,'period_fixed_profile_min_days':period,
            'max_r_hat':max_rhat,'min_effective_sample_size':min_ess,'divergences':divergences,
            'mean_accept_probability':float(np.nanmean(accept)),'median_leapfrog_steps':float(np.nanmedian(steps)),
            'fraction_at_max_tree_depth':max_tree_fraction,'seconds':round(time.time()-started,1)})
    except Exception as exc:
        case_rows.append({'simulation_row':row,'status':'failed','period_fixed_profile_min_days':period,
            'error':f'{type(exc).__name__}: {exc}','seconds':round(time.time()-started,1)})
    pd.DataFrame(case_rows).to_csv(OUT/'day9_13_nuts_case_diagnostics.csv',index=False)
    pd.DataFrame(parameter_rows).to_csv(OUT/'day9_13_per_parameter_diagnostics.csv',index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/'day9_13_conditional_intervals.csv',index=False)
    print(f'MLE-seeded conditional NUTS row {row}: {case_rows[-1]}',flush=True)

In [ ]:
if plot_rows:
    ncols=2
    nrows=int(np.ceil(len(plot_rows)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(14,4*nrows),squeeze=False,constrained_layout=True)
    for ax,(row,t,y,pred,period) in zip(axes.flat,plot_rows):
        ax.scatter(t,y,s=5,alpha=0.3,label='synthetic observations')
        ax.plot(t,pred,color='crimson',lw=0.9,label='conditional posterior median fit')
        ax.set_title(f'Row {row} | fixed period {period:.6f} d')
        ax.set_xlabel('Time (days)'); ax.set_ylabel('Flux'); ax.legend(fontsize=8)
    for ax in axes.flat[len(plot_rows):]: ax.set_visible(False)
    fig.savefig(OUT/'day9_13_conditional_fits.png',dpi=140)
    plt.show()

case_df=pd.DataFrame(case_rows)
param_df=pd.DataFrame(parameter_rows)
interval_df=pd.DataFrame(interval_rows)
passed=case_df.loc[case_df.status.eq('screen_pass'),'simulation_row'] if not case_df.empty else []
coverage={name:{'n':int(len(g)),'covered':int(g.truth_inside_90pct_interval.sum()),
                'fraction':float(g.truth_inside_90pct_interval.mean())}
          for name,g in interval_df[interval_df.simulation_row.isin(passed)].groupby('parameter')} if not interval_df.empty else {}
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'jax_backend':jax.default_backend(),
    'jax_devices':[str(d) for d in jax.devices()],'day9_12_input':str(DIAG_PATH.parent),
    'rows':CASE_ROWS,'period_fixed_from_day9_10_profile':True,
    'screen_pass_count':int(case_df.status.eq('screen_pass').sum()),
    'needs_review_or_failed_count':int(case_df.status.isin(['needs_review','failed']).sum()),
    'conditional_interval_coverage_evaluation_only':coverage,
    'sampler':{'chains':CHAINS,'warmup_per_chain':WARMUP,'samples_per_chain':SAMPLES,
        'target_accept_prob':TARGET_ACCEPT,'max_tree_depth':MAX_TREE_DEPTH,'dense_mass_blocks':[list(latent)],
        'chain_method':CHAIN_METHOD},
    'limits':['Period is fixed at a data-selected profile minimum; no period posterior is estimated.',
        'Nuisance intervals are conditional and may understate joint uncertainty.',
        'Coverage is descriptive on three selected synthetic cases, not calibration.',
        'Known truth is evaluation-only and is not used in fitting.',
        'A screen-pass does not establish correctness for real stars.'],
    'outputs':['day9_13_nuts_case_diagnostics.csv','day9_13_per_parameter_diagnostics.csv',
        'day9_13_conditional_intervals.csv','day9_13_conditional_fits.png','conditional_row_*_posterior_by_chain.npz']}
(OUT/'day9_13_nuts_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(case_df.to_string(index=False))
print('Per-parameter diagnostics:')
print(param_df.to_string(index=False))
print('Conditional coverage (evaluation-only):',json.dumps(coverage,indent=2))
print(json.dumps(summary,indent=2))
print('Outputs:',OUT)